In [0]:
%run /Users/jamiquecampbell@outlook.com/Databricks-Demos/_utils/helper_utils

In [0]:
#import standard functions
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
#date range of data to be analyzed at this layer
dbutils.widgets.text('start_date', '2025-01-01')
dbutils.widgets.text('end_date', '2025-12-01')


#calling widgets into variables
start_date = dbutils.widgets.get('start_date')
end_date = dbutils.widgets.get('end_date')
print(start_date)
print(end_date)


In [0]:
gold_df = silver_df = spark.read.table('jc_demoworkspace.datahive.silver_airline_departures')

#In this date range
gold_df = gold_df.filter(F.col('date').between(start_date, end_date))

#delayed flights only
gold_df = gold_df.filter(F.col('departure_delay_minutes') > 0)


In [0]:

gold_df = gold_df\
    .withColumn('is_delayed', F.when(F.col('departure_delay_minutes') > 0, 'Yes').otherwise('No') )\
    .withColumn('primary_delay',
                F.when(F.col('total_delay_minutes') == 0, None).otherwise(
                    F.greatest(
                        F.col('delay_carrier_minutes'),
                        F.col('delay_weather_minutes'),
                        F.col('delay_national_aviation_system_minutes'),
                        F.col('delay_security_minutes'),
                        F.col('delay_late_aircraft_arrival_minutes')
                    )
                )
    )\
    .withColumn('num_of_primary_delays',sum(
                F.when(F.col(c)==F.col('primary_delay'), 1).otherwise(0)
                for c in ['delay_carrier_minutes','delay_weather_minutes','delay_national_aviation_system_minutes','delay_security_minutes', 'delay_late_aircraft_arrival_minutes']
            )
    )\
    .withColumn('primary_delay', 
                F.when( F.col('num_of_primary_delays') > 1, 'Multiple Primary Delays')
                .when(F.col('primary_delay') == F.col('delay_carrier_minutes'), 'Carrier')
                .when(F.col('primary_delay') == F.col('delay_weather_minutes'), 'Weather')
                .when(F.col('primary_delay') == F.col('delay_national_aviation_system_minutes'), 'NAS')
                .when(F.col('primary_delay') == F.col('delay_security_minutes'), 'Security')
                .when(F.col('primary_delay') == F.col('delay_late_aircraft_arrival_minutes'), 'Late Aircraft')
                .otherwise('No Cause Defined')
    )

In [0]:
gold_ordered = gold_df.select(
    'flight_num_date_key',
    'date',
    'year',
    'month',
    'dayofweek',
    'carrier_code',
    'flight_number',
    'tail_number',
    'origin_code',
    'destination_airport',
    'scheduled_departure_timestamp',
    'actual_departure_timestamp',
    'scheduled_departure_time',
    'actual_departure_time',
    'departure_delay_minutes',
    'wheelsoff_time',
    'primary_delay',
    'total_delay_minutes',
    'delay_carrier_minutes',
    'delay_weather_minutes',
    'delay_national_aviation_system_minutes',
    'delay_security_minutes',
    'delay_late_aircraft_arrival_minutes',
    'taxiout_time_minutes'    
)

display(gold_ordered)

In [0]:
row_count = gold_ordered.count()
non_delay_rows = gold_ordered.filter(F.col('departure_delay_minutes') <= 0).count()
print(f'Total rows: {row_count}\nNon-Delayed rows: {non_delay_rows}')

#is table blank
if row_count == 0:
    raise ValueError('No data to write to table')

#non delay rows not in table check
if non_delay_rows > 0:
    raise ValueError('On-Time / Early flights included in data')

In [0]:
create_or_replace(gold_df, 'jc_demoworkspace.datahive.fact_airline_delays')